# iz-1 Pass 1 — Prithvi-EO-2.0-100M-TL fine-tune on TR-MRV-Bench

Free Colab T4 — open this notebook in Colab and run all.

**Inputs (uploaded to Drive or cloned from a private GitHub release):**
- `data/bench/train.parquet`
- `data/bench/val.parquet`
- `data/bench/test.parquet`

**Output:**
- `checkpoints/pass1_best.pt` (full-precision teacher)
- val NLL log
- per-plant MAE on test split

In [ ]:
!pip install -q torch pandas numpy pyarrow
# Optional Prithvi inference path (uncomment when satellite tiles are ready):
# !pip install -q transformers accelerate huggingface_hub

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
# Adjust path to where you put data/bench/*
BENCH = '/content/drive/MyDrive/iz/data/bench'
CKPT  = '/content/drive/MyDrive/iz/checkpoints'
import os; os.makedirs(CKPT, exist_ok=True)
!ls -lh {BENCH}

In [ ]:
import pandas as pd, numpy as np, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

NUMERIC_FEATS = [
    'no2_mean','no2_std','no2_n_pixels',
    'wind_speed_mps','wind_dir_deg','pbl_height_m','temp_2m_k',
    'thermal_mean_k','thermal_max_k','grid_co2_g_per_kwh',
]

class BenchDS(Dataset):
    def __init__(self, p):
        df = pd.read_parquet(p).dropna(subset=['co2_t_month']).reset_index(drop=True)
        X = df[NUMERIC_FEATS].astype('float32').values
        col_means = np.nan_to_num(np.nanmean(X, axis=0))
        ii = np.where(np.isnan(X)); X[ii] = np.take(col_means, ii[1])
        self.mu, self.sd = X.mean(0), X.std(0)+1e-6
        self.X = (X - self.mu) / self.sd
        self.y = np.log1p(df['co2_t_month'].astype('float32').values)
        self.w = df['label_confidence'].map({'high':1.0,'medium':0.7,'low':0.3,'':0.5}).fillna(0.5).astype('float32').values
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return torch.from_numpy(self.X[i]), torch.tensor(self.y[i]), torch.tensor(self.w[i])

class Head(nn.Module):
    def __init__(self,d=10,h=128):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(d,h), nn.GELU(), nn.Linear(h,h), nn.GELU(), nn.Dropout(0.1))
        self.mu = nn.Linear(h,1); self.ls = nn.Linear(h,1)
    def forward(self,x):
        h = self.trunk(x); return self.mu(h).squeeze(-1), self.ls(h).squeeze(-1)

def nll(mu, ls, y, w):
    s2 = torch.exp(2*ls)
    return ((0.5*((y-mu)**2/s2 + 2*ls))*w).mean()

tr = BenchDS(f'{BENCH}/train.parquet'); va = BenchDS(f'{BENCH}/val.parquet')
print('train', len(tr), 'val', len(va))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)

model = Head(d=len(NUMERIC_FEATS)).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
trL = DataLoader(tr, batch_size=32, shuffle=True); vaL = DataLoader(va, batch_size=32)
best = 1e9
for ep in range(1, 21):
    model.train()
    for x,y,w in trL:
        x,y,w = x.to(dev), y.to(dev), w.to(dev)
        mu, ls = model(x); loss = nll(mu, ls, y, w)
        opt.zero_grad(); loss.backward(); opt.step()
    model.eval()
    with torch.no_grad():
        v = float(np.mean([nll(*model(x.to(dev)), y.to(dev), w.to(dev)).item() for x,y,w in vaL]))
    print(f'ep {ep:2d}  val_nll={v:.4f}')
    if v < best:
        best = v
        torch.save({'model':model.state_dict(),'feat_mean':tr.mu,'feat_std':tr.sd}, f'{CKPT}/pass1_best.pt')
print('best val NLL:', best)

## Next: Pass 2 (ternary QAT)

Open `notebooks/pass2_colab.ipynb` after this finishes. It loads `pass1_best.pt`,
swaps Linear → BitLinear, distills from this teacher. Expect ~5% NLL bump.

## Why this is a stub for Pass 1

The full Prithvi-EO inference path expects multi-spectral satellite tiles
per-facility per-month. Until `extract_s5p_bench.py` finishes pulling those
tiles and we add Sentinel-2 + Landsat thermal + ERA5, Pass 1 trains on the
tabular feature subset only. Adding the Prithvi-EO backbone is a 30-line diff
in `iz/models/prithvi_head.py` once the tile pipeline is ready.